# Community detection

In [12]:
import networkx as nx
import pandas as pd
from networkx import community
import cdlib 


#### Load clean data

In [6]:
df_com = pd.read_csv("C:/Users/camil/Documents/IMT/3A/DaSci_UE/Graph_theory/data/all_comments.csv")

In [7]:
df_posts = pd.read_csv("C:/Users/camil/Documents/IMT/3A/DaSci_UE/Graph_theory/data/all_posts_active_subreddit.csv")

#### Constuct the graph

In [8]:
# 1. DATA PREPARATION: Link commenters to post authors
# Based on the columns in your notebook: 'Post ID' and 'Author'
df_interactions = df_com.merge(
    df_posts[['Post ID', 'Author', 'Subreddit']], 
    on='Post ID', 
    suffixes=('_commenter', '_poster')
)

# 2. GRAPH CONSTRUCTION: Building the social network
# Nodes are authors, edges represent an interaction via a comment
G = nx.from_pandas_edgelist(
    df_interactions, 
    source='Author_commenter', 
    target='Author_poster', 
    create_using=nx.Graph())

#### Identify community within subreddits

In [13]:
# 3. LOUVAIN ALGORITHM: Community Detection
# This helps identify "clusters" or "echo chambers"
# Louvain maximizes modularity to find groups with high internal density
communities = community.louvain_communities(G, seed=42)

# 4. DSA COMPLIANCE: Alert System for Community Managers
# We look for large communities with low 'Upvote Ratio' as a sign of coordinated toxicity
alerts = []
for i, comm_nodes in enumerate(communities):
    # Filter original data for users in this specific community
    comm_data = df_posts[df_posts['Author'].isin(comm_nodes)]
    
    if len(comm_nodes) > 5:  # Threshold for a "significant" group
        avg_upvote_ratio = comm_data['Upvote Ratio'].mean()
        
        # If the community has a low upvote ratio, it might be a target of a raid or bad behavior
        if avg_upvote_ratio < 0.7: 
            alerts.append({
                'Community_ID': i,
                'Member_Count': len(comm_nodes),
                'Risk_Score': 1 - avg_upvote_ratio,
                'Target_Subreddits': comm_data['Subreddit'].unique().tolist()
            })

# Print results for the Community Manager
print(f"Detected {len(communities)} distinct communities.")
print(f"Identified {len(alerts)} suspicious groups requiring DSA review.")

# Example of an alert details
if alerts:
    print(f"High Priority Alert: Community {alerts[0]['Community_ID']} is active on {alerts[0]['Target_Subreddits']}")

Detected 164 distinct communities.
Identified 0 suspicious groups requiring DSA review.
